# Material fidelity — confined uniaxial stress vs. stretch

How faithfully does each solver reproduce the **constitutive law** itself, away from
any geometry or contact? We drive a small block through the homogeneous deformation
`F = diag(1, 1, λ)` (lateral motion confined → pure uniaxial strain) and read off the
axial 1st Piola–Kirchhoff stress as `λ` sweeps from compression (0.7) to tension (1.5).

* **analytic** — the **closed-form** Neo-Hookean uniaxial-strain stress `σ(λ)`. This is
  the constitutive anchor; the same closed form is checked to machine precision by a
  test in [`tests/`](tests/test_energies.py).
* **FEM (FEniCSx)** — the implicit Neo-Hookean solve; it should trace the closed form.
* **Newton (SemiImplicit)** — Newton's **StVK / co-rotational** material. It equals the
  Neo-Hookean law at small strain (both reduce to linear elasticity at `λ ≈ 1`) and is
  expected to **depart at large strain** — this test measures exactly that gap, with no
  contact or boundary effect to confound it.

Data: `python -m newton_run.run_stress_strain` and `python -m fenics_run.run_stress_strain`.

In [ ]:
%matplotlib inline
import os

import matplotlib.pyplot as plt
import numpy as np

from common import params
from compare import energies as en
from compare import stress_strain as cmp_ss
from compare import style

fem = np.load(params.FEM_STRESS_NPZ) if os.path.exists(params.FEM_STRESS_NPZ) else None
nw = np.load(params.NEWTON_STRESS_NPZ) if os.path.exists(params.NEWTON_STRESS_NPZ) else None
print('FEM stress:', fem is not None, '| Newton stress:', nw is not None)
print(f'sweep: lambda in [{params.STRESS_LAMBDA_MIN}, {params.STRESS_LAMBDA_MAX}], '
      f'{params.STRESS_LAMBDA_N} steps; Lame mu={params.K_MU:.0f} Pa, lambda={params.K_LAMBDA:.0f} Pa')

## The scene — what we're actually simulating

A small cube (size irrelevant — the deformation is homogeneous) has the affine
displacement `u = (F − I)·X` prescribed on its **entire boundary**, with
`F = diag(1, 1, λ)`. Lateral faces are held (confined → uniaxial *strain*, not uniaxial
*stress*), and the block is stretched (`λ > 1`) or squeezed (`λ < 1`) along z. There is
no gravity, no contact: this isolates the **material law** alone.

In [ ]:
from compare import scene

nx, ny, nz = params.STRESS_DIM
h = params.STRESS_CELL
Lx, Ly, Lz = nx * h, ny * h, nz * h
fig = plt.figure(figsize=(5, 4))
ax = fig.add_subplot(111, projection='3d')
scene.draw_box(ax, (0, 0, 0), (Lx, Ly, Lz), frame_pts=[[0, 0, 0], [Lx, Ly, Lz * 1.5]])
# stretch arrows along +z (top) and -z (bottom)
ax.quiver(Lx * 0.5, Ly * 0.5, Lz, 0, 0, Lz * 0.4, color='0.3', lw=2, arrow_length_ratio=0.3)
ax.quiver(Lx * 0.5, Ly * 0.5, 0, 0, 0, -Lz * 0.4, color='0.3', lw=2, arrow_length_ratio=0.3)
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]'); ax.set_zlabel('z [m]')
ax.set_title('Setup: confined cube driven by F = diag(1, 1, lambda)')
ax.view_init(elev=18, azim=-60)
plt.tight_layout(); plt.show()

## 1. Stress vs. stretch — the constitutive curve

The axial stress against the analytic Neo-Hookean closed form. At `λ = 1` the stress is
zero (undeformed); it is negative under compression and positive under tension. FEM
should sit on the analytic curve; Newton's StVK material is expected to track it near
`λ = 1` and to peel away toward the ends.

In [ ]:
if fem is None and nw is None:
    print('No material data yet. Run: python -m newton_run.run_stress_strain '
          '(and python -m fenics_run.run_stress_strain for the FEM curve).')
else:
    fig = cmp_ss.make_stress_curve(fem, nw)
    plt.show()
    if fem is None:
        print('FEM stress curve not present yet — showing analytic + Newton only.')

## 2. Departure from the closed form vs. stretch

The signed deviation of each solver's stress from the analytic Neo-Hookean law, in kPa.
This is the **constitutive error**: ~0 near `λ = 1` (small strain, where StVK and
Neo-Hookean coincide), growing toward the compression / tension extremes if the
material model departs.

In [ ]:
lam = (fem if fem is not None else nw)['lambdas']
ana = en.neohookean_uniaxial_strain_stress(lam)

plt.figure(figsize=(6, 5))
plt.axhline(0, color=style.COLOR['analytic'], ls=style.ANALYTIC_LS, lw=1.2, label='analytic (zero error)')
if fem is not None:
    plt.plot(lam, (fem['sigma_fem'] - ana) / 1e3, 'o-', color=style.COLOR['fem'], label=style.LABEL['fem'])
if nw is not None:
    plt.plot(lam, (nw['sigma_newton'] - ana) / 1e3, 's-', color=style.COLOR['semi_implicit'],
             label=style.LABEL['semi_implicit'])
plt.axvline(1.0, color='0.7', lw=0.6)
plt.xlabel('stretch  lambda'); plt.ylabel('stress - analytic  [kPa]')
plt.title('Constitutive departure from the Neo-Hookean closed form')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

def _report(name, sig):
    dev = sig - ana
    i = int(np.argmax(np.abs(dev)))
    print(f'{name}: max |stress - analytic| = {abs(dev[i]) / 1e3:.3g} kPa at lambda={lam[i]:.2f}; '
          f'at lambda={lam[-1]:.2f}: {sig[-1] / 1e3:.3g} vs analytic {ana[-1] / 1e3:.3g} kPa')
if fem is not None:
    _report('FEM   ', fem['sigma_fem'])
if nw is not None:
    _report('Newton', nw['sigma_newton'])

## Takeaway

* The **material law** is the cleanest test in the suite: no geometry, no contact, just
  `σ(λ)` against a closed form. The analytic Neo-Hookean stress is the anchor (checked to
  machine precision in [`tests/`](tests/test_energies.py)); the FEM solve is expected to
  reproduce it, confirming the constitutive implementation.
* **Newton's StVK / co-rotational** material coincides with Neo-Hookean at small strain
  and the deviation in §2 quantifies where it departs under larger stretch / compression
  — the material-model half of the overall Newton-vs-FEM gap, isolated from the
  solver-equilibrium half measured in the hanging-bar and convergence scenarios.